In [19]:
import time
import copy
import numpy as np

from numpy.random import random, shuffle
from scipy.stats import norm
from scipy.optimize import minimize
from warnings import catch_warnings, simplefilter
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import WhiteKernel, ConstantKernel as C , RBF, Matern, RationalQuadratic, DotProduct

In [20]:
# import time
# import numpy as np

# from numpy.random import normal
# from numpy.random import random
# from numpy.random import randint
# from math import pi

# from sklearn.gaussian_process import GaussianProcessRegressor
# #from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel as C, RBF
# from sklearn.gaussian_process.kernels import WhiteKernel, ConstantKernel as C , RBF, Matern, RationalQuadratic, DotProduct

# from scipy import optimize

In [21]:
from numpy import pi


def cae_Goldstein_Price(x, w_imp=1.0, w_dum=0.001):
    # Goldstein-Price function for x[0], x[1], plus extra terms for x[2], x[3], x[4]

    x = np.asarray(x)

    term1 = 1 + (x[0] + x[1] + 1)**2 * (19 - 14*x[0] + 3*x[0]**2 - 14*x[1] + 6*x[0]*x[1] + 3*x[1]**2)
    term2 = 30 + (2*x[0] - 3*x[1])**2 * (18 - 32*x[0] + 12*x[0]**2 + 48*x[1] - 36*x[0]*x[1] + 27*x[1]**2)

    f_gp = term1 * term2

    if x.size > 2:
        term_imp = w_imp * x[2]**2
        term_dum = w_dum * (x[3]**2 + x[4]**2)

    else:
        term_imp = 0.0
        term_dum = 0.0

    f_cae = f_gp + term_imp + term_dum
        
    return f_cae


def cae_Rosenbrocks_Valley(x, w_imp=1.0, w_dum=0.001):
    # n=5 Rosenbrock's Valley function for x[0]..x[4]
    # x[5] is treated as important (imp), x[6], x[7] are dummy
    x = np.asarray(x)
    
    xb = x[:5]
    rosen = np.sum(100.0*(xb[1:] - xb[:-1]**2)**2 + (1.0 - xb[:-1])**2)  + 1.0 # to make the minimum value 1.0

    if x.size > 5:
        x6 = x[5] if x.size > 5 else 0.0 # important variable
        x7 = x[6] if x.size > 6 else 0.0 # dummy variable
        x8 = x[7] if x.size > 7 else 0.0 # dummy variable
        term_imp = w_imp * x6**2
        term_dum = w_dum * (x7**2 + x8**2)
    else:
        term_imp = 0.0
        term_dum = 0.0
        
    f_cae = rosen + term_imp + term_dum    
    return f_cae

def cae_Six_Hump_Camel_Back(x, w_imp=1.0, w_dum=0.001):
    # Six Hump Camel Back Function
    x = np.asarray(x)

    term1 = (4 - 2.1*x[0]**2 + (x[0]**4)/3) * x[0]**2
    term2 = x[0]*x[1]
    term3 = (-4 + 4*x[1]**2) * x[1]**2
    
    if x.size > 2:
        term_imp = w_imp * x[2]**2
        term_dum = w_dum * (x[3]**2 + x[4]**2)

    else:
        term_imp = 0.0
        term_dum = 0.0
    
    f_cae = term1 + term2 + term3 + term_imp + term_dum
    
    return f_cae

def cae_Cantilever_Beam(x, w_imp=1.0, w_dum=0.001):
    """
    Cantilever Beam (continuous taper) objective and list_constraints.
    x is expected to contain at least 7 entries:
      x[0]=H, x[1]=h1, x[2]=b1, x[3]=b2,
      x[4]=dum1,  x[5]=dum2,  x[6]=dum3
    Returns: (f_cae, sigma_max, delta_max)
    - f_cae: objective (end deflection delta_end + penalties on dummy vars)
    - sigma_max: maximum bending stress along the beam
    - delta_max: deflection at free end (delta_end)
    """
    # Physical / numerical constants
    L = 36.0          # beam length (m)
    P = 1000.0          # end point load (N) (can be scaled externally)
    E = 10.0e6       # Young's modulus (Pa), typical steel


    x = np.asarray(x)

    # unpack x with safe defaults for shorter vectors
    H = float(x[0]) if len(x) > 0 else 0.0
    h1   = float(x[1]) if len(x) > 1 else H
    b1  = float(x[2]) if len(x) > 2 else 0.01
    b2   = float(x[3]) if len(x) > 3 else b1
    dum1    = float(x[4]) if len(x) > 4 else 0.0
    dum2    = float(x[5]) if len(x) > 5 else 0.0
    dum3    = float(x[6]) if len(x) > 6 else 0.0

    # Prevent non-physical or zero cross-sections
    eps = 1e-9
    H = max(abs(H), eps)
    h1   = max(abs(h1), eps)
    b1  = max(abs(b1), eps)
    b2   = max(abs(b2), eps)

    V = (2*h1*b1 + (H - 2*h1) * b2 ) * L  # beam volume
    
    I_moment = (1./12)*b2*(H - 2*h1)**3 + 2*((1./12)*b1*h1**3 + (1./4)*b1*h1*(H - h1)**2)# approx. moment of inertia at root

    # bending stress (at outer fiber y = h/2): sigma = M * (h/2) / I = 6*M/(b*h^2)
    sigma_max = P*L*H/(2*I_moment)  # max bending moment at root M = P*L; sigma_max <= 5,000

    # deflection at free end for tapered beam:
    delta_max = P*(L**3)/(3*E*I_moment) # delta_max <= 0.10    
    
    if x.size > 4:
        # objective: minimize tip deflection + penalties on dummy variables
        term_imp = w_imp * dum1**2
        term_dum = w_dum * (dum2**2 + dum3**2)
        
    else:
        # objective: minimize tip deflection + penalties on dummy variables
        term_imp = 0.0
        term_dum = 0.0
        
    f_cae = float(V + term_imp + term_dum)


    return f_cae, sigma_max, delta_max

def cae_My_Function(x, w_imp=1.0, w_dum=0.001):
    # 
    # n_para = 4
    # n_para_important = 3

    # lb_const=0 # -1 
    # ub_const=+1 # +1 

    # lb=lb_const*np.ones(n_para)
    # ub=ub_const*np.ones(n_para)

    # f_weight = np.ones(n_para)
    # if n_para > n_para_important:
    #     f_weight[:n_para_important] = 1*f_weight[:n_para_important]
    #     f_weight[n_para_important:] = 1e-8*f_weight[n_para_important:]

    # x_true = 0.9015*np.ones(n_para)
    # y_true = np.sum(f_weight*(-0.8113496993537722*np.ones(n_para)))    
        
    x = np.asarray(x)
    
    n_para = np.size(x)
    
    f_cae = 0
    for i in range(n_para):
        f_cae += (x[i]**2 * np.sin(5 * pi * x[i])**6.0)
    f_cae=-f_cae    
        
    return f_cae




In [22]:
def objective(x, args=[]):
    # Goldstein-Price function for x[0], x[1], plus extra terms for x[2], x[3], x[4]
    
    global cnt_objective
    cnt_objective += 1
    
    cae_tool_name = args[0] 
    
    if cae_tool_name == 'Goldstein_Price':
        obj_value = cae_Goldstein_Price(x, w_imp=1.0, w_dum=0.001)
    elif cae_tool_name == 'Rosenbrocks_Valley':
        obj_value = cae_Rosenbrocks_Valley(x, w_imp=1.0, w_dum=0.001)
    elif cae_tool_name == 'Six_Hump_Camel_Back':
        obj_value = cae_Six_Hump_Camel_Back(x, w_imp=1.0, w_dum=0.001)
    elif cae_tool_name == 'Cantilever_Beam':
        obj_value, sigma_max, delta_max = cae_Cantilever_Beam(x, w_imp=1.0, w_dum=0.001)
        #obj_value = obj_value + 0.001*(1*max(0, sigma_max-5000)**2 + 1e3*max(0, delta_max-0.1)**2)
        #obj_value = obj_value + 1e3*(1*max(0, sigma_max-5000)**2 + 1e3*max(0, delta_max-0.1)**2)
        obj_value = obj_value + 0*(1*max(0, sigma_max-5000)**2 + 1e3*max(0, delta_max-0.1)**2)
    elif cae_tool_name == 'My_Function':
        obj_value = cae_My_Function(x, w_imp=1.0, w_dum=0.001)
        
    else:
        raise ValueError(f"Unknown CAE model name: {cae_tool_name}")
    return obj_value

In [23]:
def constraints(x, args=[]):
    
    # Note: cnt_objective is counted in objective() function, not here
    # to avoid double counting
    
    cae_tool_name = args[0]     
    
    if cae_tool_name == 'Cantilever_Beam':
        _, sigma_max, delta_max = cae_Cantilever_Beam(x, w_imp=1.0, w_dum=0.001)
        ineq_con = [sigma_max - 5000, delta_max - 0.1]  # ineq_con < 0
        #ineq_con = [sigma_max - 5100, delta_max - 0.1]  # ineq_con < 0
        #ineq_con = []
    elif cae_tool_name == 'My_Function':
        ineq_con = [] #[0.6 - x[i] for i in range(len(x))] # ineq_con < 0
    else:       
        ineq_con = []
    
    #ineq_con = [] # ineq_con < 0
    #ineq_con = [list(0.6 - x[:5])] # ineq_con < 0
    #ineq_con = [0.5 - x[0], 0.5 - x[1], 0.5 - x[2], 0.5 - x[3], 0.5 - x[4]] # ineq_con < 0
    
    return ineq_con

In [24]:
def constraints_old(x, args=[]):
    
    # Note: 평가 횟수를 절약하기 위해 제약조건 체크를 건너뜀 
    # Cantilever_Beam의 경우 objective 함수에서 penalty로 처리됨
    
    cae_tool_name = args[0]     
    
    if cae_tool_name == 'Cantilever_Beam':
        # 제약조건 체크 없이 모든 샘플 허용 (objective에서 penalty 처리)
        ineq_con = []  
    else:       
        ineq_con = []
    
    return ineq_con

In [25]:
from sklearn.gaussian_process import GaussianProcessRegressor
#from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel as C, RBF
from sklearn.gaussian_process.kernels import WhiteKernel, ConstantKernel as C , RBF, Matern, RationalQuadratic, DotProduct

# Suppress sklearn GP warnings for numerical stability
import warnings
warnings.filterwarnings('ignore', message='.*Predicted variances smaller than 0.*', module='sklearn')
warnings.filterwarnings('ignore', category=Warning, module='sklearn')
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings('ignore', category=ConvergenceWarning)

def _scale_args(d: int, noise: float = 1e-6, L_kernel: float = 1.0, L_scale_const: float = 0.01):
    """
    수치적 안정성을 위한 개선된 스케일 인자 계산
    """
    L_scale = L_scale_const * L_kernel
    # 더 보수적인 bounds로 수치적 안정성 확보
    const_bounds = (max(L_scale, 1e-3), min(1.0 / L_scale, 1e3))  # 극단값 제한
    ard_bounds = [const_bounds] * d
    return L_scale, const_bounds, ard_bounds

def kernel_common_best(d: int, noise: float = 1e-6, L_kernel: float = 1.0, L_scale_const: float = 0.01):
    """
    수치적 안정성이 개선된 최고 성능 커널
    """
    L_scale, const_bounds, ard_bounds = _scale_args(d, noise, L_kernel, L_scale_const)
    
    # 안전한 노이즈 레벨 보장
    safe_noise = max(noise, 1e-8)  # 최소 노이즈 레벨 보장
    
    kernel = (
        C(constant_value=1.0 * L_kernel, constant_value_bounds=const_bounds)
        * Matern(length_scale=[1.0 * L_kernel] * d, length_scale_bounds=ard_bounds, nu=2.5)
        + WhiteKernel(noise_level=safe_noise, noise_level_bounds=(1e-8, 1e-2))  # 안전한 범위
    )
    return kernel

def kernel_common_smooth(d: int, noise: float = 1e-6, L_kernel: float = 1.0, L_scale_const: float = 0.01):
    """
    수치적 안정성이 개선된 부드러운 커널
    """
    L_scale, const_bounds, ard_bounds = _scale_args(d, noise, L_kernel, L_scale_const)
    safe_noise = max(noise, 1e-8)
    
    kernel = (
        C(constant_value=1.0 * L_kernel, constant_value_bounds=const_bounds)
        * RBF(length_scale=[1.0 * L_kernel] * d, length_scale_bounds=ard_bounds)
        + WhiteKernel(noise_level=safe_noise, noise_level_bounds=(1e-8, 1e-2))
    )
    return kernel

def kernel_common_composite(d: int, noise: float = 1e-6, L_kernel: float = 1.0, L_scale_const: float = 0.01):
    L_scale, const_bounds, ard_bounds = _scale_args(d, noise, L_kernel, L_scale_const)
    safe_noise = max(noise, 1e-8)
    
    kernel = (
        C(constant_value=1.0 * L_kernel, constant_value_bounds=const_bounds)
        * (
            Matern(length_scale=[1.0 * L_kernel] * d, length_scale_bounds=ard_bounds, nu=2.5)
            + RationalQuadratic(length_scale=1.0 * L_kernel, alpha=1.0)
        )
        + WhiteKernel(noise_level=safe_noise, noise_level_bounds=(1e-8, 1e-2))
    )
    return kernel

def kernel_additive_like(d: int, noise: float = 1e-6, L_kernel: float = 1.0, L_scale_const: float = 0.01):
    L_scale, const_bounds, ard_bounds = _scale_args(d, noise, L_kernel, L_scale_const)
    safe_noise = max(noise, 1e-8)
    
    kernel = (
        C(constant_value=1.0 * L_kernel, constant_value_bounds=const_bounds)
        * (
            Matern(length_scale=[1.0 * L_kernel] * d, length_scale_bounds=ard_bounds, nu=2.5)
            + DotProduct(sigma_0=1.0)
        )
        + WhiteKernel(noise_level=safe_noise, noise_level_bounds=(1e-8, 1e-2))
    )
    return kernel

def kernel_speed_performance_balanced(d: int, noise: float = 1e-6, L_kernel: float = 1.0, L_scale_const: float = 0.01):
    L_scale, const_bounds, ard_bounds = _scale_args(d, noise, L_kernel, L_scale_const)
    safe_noise = max(noise, 1e-8)
    
    kernel = (
        C(constant_value=1.0 * L_kernel, constant_value_bounds=const_bounds)
        * (
            Matern(length_scale=[1.0 * L_kernel] * d, length_scale_bounds=ard_bounds, nu=2.5)
            + 0.25 * DotProduct(sigma_0=1.0)
        )
        + WhiteKernel(noise_level=safe_noise, noise_level_bounds=(1e-8, 1e-2))
    )
    return kernel

def kernel_speed_performance_balanced_fast(d: int, noise: float = 1e-6, L_kernel: float = 1.0, L_scale_const: float = 0.01):
    L_scale, const_bounds, _ = _scale_args(d, noise, L_kernel, L_scale_const)
    safe_noise = max(noise, 1e-8)
    
    kernel = (
        C(constant_value=1.0 * L_kernel, constant_value_bounds=const_bounds)
        * (
            Matern(length_scale=1.0 * L_kernel, length_scale_bounds=const_bounds, nu=2.5)
            + 0.25 * DotProduct(sigma_0=1.0)
        )
        + WhiteKernel(noise_level=safe_noise, noise_level_bounds=(1e-8, 1e-2))
    )
    return kernel

def kernel_speed_performance_balanced_smooth(d: int, noise: float = 1e-6, L_kernel: float = 1.0, L_scale_const: float = 0.01):
    L_scale, const_bounds, ard_bounds = _scale_args(d, noise, L_kernel, L_scale_const)
    safe_noise = max(noise, 1e-8)
    
    kernel = (
        C(constant_value=1.0 * L_kernel, constant_value_bounds=const_bounds)
        * (
            RBF(length_scale=[1.0 * L_kernel] * d, length_scale_bounds=ard_bounds)
            + 0.25 * DotProduct(sigma_0=1.0)
        )
        + WhiteKernel(noise_level=safe_noise, noise_level_bounds=(1e-8, 1e-2))
    )
    return kernel

def kernel_speed_linear(d: int, noise: float = 1e-6, L_kernel: float = 1.0, L_scale_const: float = 0.01):
    L_scale, const_bounds, _ = _scale_args(d, noise, L_kernel, L_scale_const)
    safe_noise = max(noise, 1e-8)
    
    kernel = (
        C(constant_value=1.0 * L_kernel, constant_value_bounds=const_bounds) * DotProduct(sigma_0=1.0)
        + WhiteKernel(noise_level=safe_noise, noise_level_bounds=(1e-8, 1e-2))
    )
    return kernel

def kernel_speed_rbf_iso(d: int, noise: float = 1e-6, L_kernel: float = 1.0, L_scale_const: float = 0.01):
    L_scale, const_bounds, _ = _scale_args(d, noise, L_kernel, L_scale_const)
    safe_noise = max(noise, 1e-8)
    
    kernel = (
        C(constant_value=1.0 * L_kernel, constant_value_bounds=const_bounds)
        * RBF(length_scale=1.0 * L_kernel, length_scale_bounds=const_bounds)
        + WhiteKernel(noise_level=safe_noise, noise_level_bounds=(1e-8, 1e-2))
    )
    return kernel

def kernel_speed_matern32_iso(d: int, noise: float = 1e-6, L_kernel: float = 1.0, L_scale_const: float = 0.01):
    L_scale, const_bounds, _ = _scale_args(d, noise, L_kernel, L_scale_const)
    safe_noise = max(noise, 1e-8)
    
    kernel = (
        C(constant_value=1.0 * L_kernel, constant_value_bounds=const_bounds)
        * Matern(length_scale=1.0 * L_kernel, length_scale_bounds=const_bounds, nu=1.5)
        + WhiteKernel(noise_level=safe_noise, noise_level_bounds=(1e-8, 1e-2))
    )          
    return kernel

def kernel_speed_matern52_iso(d: int, noise: float = 1e-6, L_kernel: float = 1.0, L_scale_const: float = 0.01):
    L_scale, const_bounds, _ = _scale_args(d, noise, L_kernel, L_scale_const)
    safe_noise = max(noise, 1e-8)
    
    kernel = (
        C(constant_value=1.0 * L_kernel, constant_value_bounds=const_bounds)
        * Matern(length_scale=1.0 * L_kernel, length_scale_bounds=const_bounds, nu=2.5)
        + WhiteKernel(noise_level=safe_noise, noise_level_bounds=(1e-8, 1e-2))
    )
    return kernel

def kernel_speed_linear_plus_rbf_iso(d: int, noise: float = 1e-6, L_kernel: float = 1.0, L_scale_const: float = 0.01):
    L_scale, const_bounds, _ = _scale_args(d, noise, L_kernel, L_scale_const)
    safe_noise = max(noise, 1e-8)
    
    kernel = (
        C(constant_value=1.0 * L_kernel, constant_value_bounds=const_bounds)
        * (DotProduct(sigma_0=1.0) + RBF(length_scale=1.0 * L_kernel, length_scale_bounds=const_bounds))
        + WhiteKernel(noise_level=safe_noise, noise_level_bounds=(1e-8, 1e-2))
    )
    return kernel

# Example of the specific form you showed (two RBFs with scaled initial lengths and bounds)
def kernel_example_two_rbfs(n_para: int, noise: float = 1e-6, L_kernel: float = 1.0, L_scale_const: float = 0.01):
    L_scale, const_bounds, ard_bounds = _scale_args(n_para, noise, L_kernel, L_scale_const)
    safe_noise = max(noise, 1e-8)
    
    kernel = (
        C(constant_value=1.0 * L_kernel, constant_value_bounds=const_bounds)
        * RBF(length_scale=[0.5 * L_kernel] * n_para, length_scale_bounds=ard_bounds)
        + RBF(length_scale=[2.0 * L_kernel] * n_para, length_scale_bounds=ard_bounds)
    )
    return kernel

# ===================================================================
# 🛡️ NUMERICALLY STABLE KERNELS FOR ROBUST OPTIMIZATION
# ===================================================================

def kernel_stable_balanced(d: int, noise: float = 1e-6, L_kernel: float = 1.0, L_scale_const: float = 0.1):
    """
    수치적 안정성에 최적화된 균형잡힌 커널
    - 보수적인 파라미터 범위
    - 안전한 노이즈 레벨
    - 견고한 커널 조합
    """
    safe_noise = max(noise, 1e-7)  # 안전한 최소 노이즈
    
    # 보수적인 bounds 설정
    safe_const_bounds = (0.01, 100.0)  # 극단값 방지
    safe_length_bounds = [(0.01, 10.0)] * d
    
    kernel = (
        # 안정적인 주 커널 (Matern 2.5)
        C(1.0, safe_const_bounds) * Matern(
            length_scale=[1.0] * d, 
            length_scale_bounds=safe_length_bounds, 
            nu=2.5
        ) +
        # 보조 커널 (더 부드러운 RBF)
        C(0.1, (0.01, 10.0)) * RBF(
            length_scale=[0.5] * d, 
            length_scale_bounds=safe_length_bounds
        ) +
        # 안전한 노이즈 커널
        WhiteKernel(noise_level=safe_noise, noise_level_bounds=(1e-7, 1e-2))
    )
    return kernel

def kernel_stable_conservative(d: int, noise: float = 1e-6, L_kernel: float = 1.0, L_scale_const: float = 0.1):
    """
    최고 수치적 안정성을 위한 보수적 커널
    - 단일 안정적 커널 사용
    - 넓은 노이즈 여유
    - 제한된 파라미터 범위
    """
    safe_noise = max(noise, 1e-6)  # 더 높은 안전 노이즈
    
    kernel = (
        C(1.0, (0.1, 10.0)) * Matern(
            length_scale=[1.0] * d, 
            length_scale_bounds=[(0.1, 5.0)] * d, 
            nu=2.5
        ) +
        WhiteKernel(noise_level=safe_noise, noise_level_bounds=(1e-6, 1e-1))
    )
    return kernel

# ===================================================================
# 🚀 IMPROVED ROSENBROCK KERNELS WITH NUMERICAL STABILITY
# ===================================================================

def kernel_high_dim_stable(d: int, noise: float = 1e-6, L_kernel: float = 1.0, L_scale_const: float = 0.1):
    """
    Rosenbrock 함수에 특화되었지만 수치적으로 안정한 커널
    """
    safe_noise = max(noise, 1e-7)
    
    # 안전한 bounds로 조정
    safe_bounds = [(0.05, 8.0)] * d
    const_bounds = (0.05, 20.0)
    
    kernel = (
        # 장거리 상관관계 (안정한 Matern 2.5)
        C(0.6, const_bounds) * Matern(
            length_scale=[2.0] * d, 
            length_scale_bounds=safe_bounds, 
            nu=2.5
        ) +
        # 중간거리 구조 (안정한 RBF)
        C(0.3, (0.05, 10.0)) * RBF(
            length_scale=[0.8] * d, 
            length_scale_bounds=safe_bounds
        ) +
        # 지역 구조 (안정한 Matern 1.5)
        C(0.1, (0.01, 5.0)) * Matern(
            length_scale=[0.3] * d, 
            length_scale_bounds=safe_bounds, 
            nu=1.5
        ) +
        # 안전한 노이즈
        WhiteKernel(noise_level=safe_noise, noise_level_bounds=(1e-7, 1e-2))
    )
    return kernel

def kernel_high_dim_ultimate_stable(d: int, noise: float = 1e-6, L_kernel: float = 1.0, L_scale_const: float = 0.1):
    """
    속도 최적화된 Rosenbrock 커널 (2-level, 10배 빠름)
    """
    safe_noise = max(noise, 1e-7)
    
    # 보수적이지만 효과적인 bounds
    safe_bounds = [(0.1, 10.0)] * d
    const_bounds = (0.1, 50.0)
    
    kernel = (
        # Level 1: 전역 구조 (Matern 2.5)
        C(0.7, const_bounds) * Matern(
            length_scale=[2.0] * d, 
            length_scale_bounds=safe_bounds, 
            nu=2.5
        ) +
        # Level 2: 지역 구조 (RBF)
        C(0.3, (0.05, 10.0)) * RBF(
            length_scale=[0.5] * d, 
            length_scale_bounds=safe_bounds
        ) +
        # 안전한 노이즈
        WhiteKernel(noise_level=safe_noise, noise_level_bounds=(1e-7, 1e-2))
    )
    return kernel

In [ ]:
# ===================================================================
# 🎯 ACQUISITION FUNCTION HELPERS
# ===================================================================

def acquisition_lcb(X, model, kappa=2.0):
    """
    Lower Confidence Bound acquisition function
    """
    mu, sigma = model.predict(X.reshape(1, -1), return_std=True)
    return mu - kappa * sigma

def acquisition_ei(X, model, y_best, xi=0.01): # Original Setup: xi=0.01, 0.1[NG]
    """
    Expected Improvement acquisition function
    """
    mu, sigma = model.predict(X.reshape(1, -1), return_std=True)
    sigma = sigma.reshape(-1, 1)
    
    with np.errstate(divide='warn'):
        imp = y_best - mu - xi
        Z = imp / (sigma + 1e-9)
        ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
        ei[sigma == 0.0] = 0.0
    
    return -ei  # Minimize for scipy

def optimize_acquisition(model, y_best, lb, ub, n_restarts=10, acq_type='LCB', kappa=2.0):
    """
    Global optimization of acquisition function
    """
    from scipy.optimize import minimize
    
    #dim = len(lb)
    best_x = None
    best_acq = np.inf
    
    for _ in range(n_restarts):
        x0 = np.random.uniform(lb, ub)
        
        if acq_type == 'LCB':
            result = minimize(
                lambda x: acquisition_lcb(x, model, kappa),
                x0, method='L-BFGS-B', bounds=list(zip(lb, ub))
            )
        else:  # EI
            result = minimize(
                lambda x: acquisition_ei(x, model, y_best),
                x0, method='L-BFGS-B', bounds=list(zip(lb, ub))
            )
        
        if result.fun < best_acq:
            best_acq = result.fun
            best_x = result.x
    
    return best_x

# ===================================================================
# 🚀 GP-BASED ADAPTIVE OPTIMIZATION WITH ACQUISITION FUNCTIONS
# Intelligent optimization using surrogate model and learned patterns
# ===================================================================

def gp_adaptive_optimization(objective, constraints, kernel, lb, ub, args, max_eval_objective, threshold=200, early_stop_y_tolerance=-np.inf, early_stop_patience=np.inf):
    """
    GP surrogate model과 acquisition function을 활용한 스마트 적응형 최적화
    Valley 구조 문제에 특화된 커널과 탐색 전략 사용
    
    Parameters:
        early_stop_y_tolerance: 목표 y 값. best_y <= early_stop_y_tolerance 이면 조기 종료
        early_stop_patience: 연속으로 개선이 없는 평가 횟수. 이 횟수만큼 개선 없으면 조기 종료
    """
    #from scipy.optimize import differential_evolution
    
    # 로컬 평가 횟수 카운터
    local_eval_count = 0
    n_dim = len(lb)
    
    # 조기 종료 카운터
    no_improve_count = 0
    
    def _check_early_stop(best_y, no_improve_count):
        """early_stop_y_tolerance 또는 early_stop_patience 조건 확인"""
        if best_y <= early_stop_y_tolerance:
            print(f"🎉 Goal reached! best_y={best_y:.6f} <= early_stop_y_tolerance={early_stop_y_tolerance:.6f}")
            return True
        if no_improve_count >= early_stop_patience:
            print(f"🛑 Early stopping! No improvement for {no_improve_count} consecutive evaluations (patience={early_stop_patience})")
            return True
        return False
    
    # ===================================================================
    # 🎯 PROBLEM CHARACTERISTIC DETECTION & KERNEL SELECTION
    # ===================================================================
    is_high_dim_intensive = (n_dim >= 4 and max_eval_objective >= 1000)
    
    if is_high_dim_intensive:
        print(f"🔍 Detected high-dimensional intensive problem (dim={n_dim}, evals={max_eval_objective})")
        print(f"   Using valley-specialized kernel for better performance")
        # Valley 특화 커널 사용
        kernel = kernel_high_dim_ultimate_stable(n_dim)
    else:
        # 일반 고성능 커널
        kernel = kernel_common_best(n_dim)
    
    if max_eval_objective >= threshold:
        # 충분한 평가 횟수: GP 기반 다단계 적응형 최적화
        print(f"📈 GP-based multi-phase optimization (evals: {max_eval_objective} >= {threshold})")
        
        # ===================================================================
        # 🔍 PHASE 1: INITIAL EXPLORATION + GP FITTING (25%)
        # ===================================================================
        phase1_evals = int(max_eval_objective * 0.25)
        print(f"🔍 Phase 1: Initial exploration + GP fitting with {phase1_evals} evaluations")
        
        X_all = []
        y_all = []
        best_x = None
        best_y = np.inf
        early_stopped = False
        
        # Latin Hypercube Sampling for better space coverage
        for i in range(phase1_evals):
            if local_eval_count >= max_eval_objective:
                break
            
            # Diversified sampling strategies
            if i < phase1_evals // 3:
                # Pure random
                x_sample = np.array(lb) + np.random.random(n_dim) * (np.array(ub) - np.array(lb))
            elif i < 2 * phase1_evals // 3:
                # Sobol-like sampling (split each dimension)
                x_sample = np.array([lb[j] + (i % 5 + np.random.random()) / 5 * (ub[j] - lb[j]) for j in range(n_dim)])
                x_sample = np.clip(x_sample, lb, ub)
            else:
                # Center-biased sampling
                center = np.array([(lb[j] + ub[j])/2 for j in range(n_dim)])
                x_sample = center + 0.4 * np.random.randn(n_dim) * (np.array(ub) - np.array(lb))
                x_sample = np.clip(x_sample, lb, ub)
            
            # 제약조건 체크
            if constraints is not None:
                list_constraints = constraints(x_sample, args)
                if list_constraints and any(c > 0 for c in list_constraints):
                    continue
            
            try:
                y_sample = objective(x_sample, args)
                local_eval_count += 1
                X_all.append(x_sample)
                y_all.append(y_sample)
                
                if y_sample < best_y:
                    best_y = y_sample
                    best_x = x_sample.copy()
                    no_improve_count = 0
                else:
                    no_improve_count += 1
                
                if _check_early_stop(best_y, no_improve_count):
                    early_stopped = True
                    break
                    
            except Exception as e:
                continue
        
        if len(X_all) == 0:
            # 최후 수단
            best_x = np.array([(lb[i] + ub[i])/2 for i in range(n_dim)])
            best_y = objective(best_x, args)
            local_eval_count += 1
            X_all = [best_x]
            y_all = [best_y]
        
        X_all = np.array(X_all)
        y_all = np.array(y_all)
        print(f"Phase 1 complete. Best: {best_y:.4f} from {len(X_all)} samples")
        
        if early_stopped:
            history = {
                'method': 'gp_multi_phase',
                'samples': local_eval_count,
                'kernel': 'valley_specialized' if is_high_dim_intensive else 'common_best',
                'early_stopped': True,
                'stopped_at': 'phase1'
            }
            return best_x, best_y, history
        
        # ===================================================================
        # 🤖 FIT GP SURROGATE MODEL
        # ===================================================================
        print(f"🤖 Fitting GP surrogate model...")
        try:
            gp_model = GaussianProcessRegressor(
                kernel=kernel,
                alpha=1e-6,
                normalize_y=True, #False,
                n_restarts_optimizer=1,
                random_state=rand_seed
            )
            gp_model.fit(X_all, y_all)
            print(f"   GP model fitted successfully")
        except Exception as e:
            print(f"   Warning: GP fitting failed ({e}), using fallback kernel")
            kernel_fallback = kernel_stable_conservative(n_dim)
            gp_model = GaussianProcessRegressor(
                kernel=kernel_fallback,
                alpha=1e-5,
                normalize_y=True, #False,
                n_restarts_optimizer=1
            )
            gp_model.fit(X_all, y_all)
        
        # ===================================================================
        # 🎯 PHASE 2: ACQUISITION-BASED EXPLOITATION (45%)
        # ===================================================================
        phase2_evals = int(max_eval_objective * 0.45)
        print(f"🎯 Phase 2: Acquisition-based exploitation with {phase2_evals} evaluations")
        
        for i in range(phase2_evals):
            if local_eval_count >= max_eval_objective:
                break
            
            # Adaptive kappa: start explorative, become exploitative
            progress = i / phase2_evals
            kappa = 3.0 - 2.5 * progress  # 3.0 → 0.5
            
            # 40% acquisition-based, 60% random exploration (속도 최적화)
            if np.random.random() < 0.4:
                try:
                    # LCB acquisition function으로 다음 샘플 찾기
                    x_sample = optimize_acquisition(
                        gp_model, best_y, lb, ub, 
                        n_restarts=2, acq_type='LCB', kappa=kappa
                    )
                except:
                    # Fallback: random near best point
                    x_sample = best_x + 0.1 * np.random.randn(n_dim) * (np.array(ub) - np.array(lb))
                    x_sample = np.clip(x_sample, lb, ub)
            else:
                # Random exploration near promising regions
                n_top = max(3, len(y_all) * 20 // 100)
                top_indices = np.argsort(y_all)[:n_top]
                center = X_all[top_indices[np.random.randint(len(top_indices))]]
                radius = 0.15 * (1 - 0.5 * progress)  # 0.15 → 0.075
                x_sample = center + radius * np.random.randn(n_dim) * (np.array(ub) - np.array(lb))
                x_sample = np.clip(x_sample, lb, ub)
            
            # 제약조건 체크
            if constraints is not None:
                list_constraints = constraints(x_sample, args)
                if list_constraints and any(c > 0 for c in list_constraints):
                    continue
            
            try:
                y_sample = objective(x_sample, args)
                local_eval_count += 1
                X_all = np.vstack([X_all, x_sample])
                y_all = np.append(y_all, y_sample)
                
                if y_sample < best_y:
                    best_y = y_sample
                    best_x = x_sample.copy()
                    no_improve_count = 0
                    print(f"   ⭐ New best at eval {local_eval_count}: {best_y:.4f}")
                else:
                    no_improve_count += 1
                
                if _check_early_stop(best_y, no_improve_count):
                    early_stopped = True
                    break
                
                # Update GP model periodically (속도 최적화: 덜 자주 업데이트)
                #if i > 0 and i % 100 == 0:
                if i > 0 and i % int(max_eval_objective * 0.1) == 0:
                    try:
                        gp_model.fit(X_all, y_all)
                    except:
                        pass
                    
            except Exception as e:
                continue
        
        print(f"Phase 2 complete. Best: {best_y:.4f}")
        
        if early_stopped:
            history = {
                'method': 'gp_multi_phase',
                'samples': local_eval_count,
                'kernel': 'valley_specialized' if is_high_dim_intensive else 'common_best',
                'early_stopped': True,
                'stopped_at': 'phase2'
            }
            return best_x, best_y, history
        
        # ===================================================================
        # 🤖 REFIT GP MODEL WITH ALL DATA
        # ===================================================================
        try:
            gp_model.fit(X_all, y_all)
            print(f"🤖 GP model refitted with {len(X_all)} samples")
        except:
            pass
        
        # ===================================================================
        # 🔬 PHASE 3: GP-GUIDED LOCAL REFINEMENT (30%)
        # ===================================================================
        phase3_evals = max_eval_objective - local_eval_count
        print(f"🔬 Phase 3: GP-guided local refinement with {phase3_evals} evaluations")
        
        for i in range(phase3_evals):
            if local_eval_count >= max_eval_objective:
                break
            
            progress = i / max(1, phase3_evals)
            
            # 80% exploitation near best, 20% acquisition-guided
            if np.random.random() < 0.8:
                # Local search near current best with decreasing radius
                if i < phase3_evals // 2:
                    radius = 0.05 - 0.03 * progress  # 0.05 → 0.02
                else:
                    radius = 0.02 - 0.015 * progress  # 0.02 → 0.005
                
                direction = np.random.randn(n_dim)
                direction = direction / (np.linalg.norm(direction) + 1e-9)
                x_sample = best_x + radius * (np.array(ub) - np.array(lb)) * direction
                x_sample = np.clip(x_sample, lb, ub)
            else:
                # Acquisition-guided refinement (높은 exploitation)
                try:
                    kappa = 0.5  # Very low kappa for pure exploitation
                    x_sample = optimize_acquisition(
                        gp_model, best_y, lb, ub,
                        n_restarts=3, acq_type='LCB', kappa=kappa
                    )
                except:
                    x_sample = best_x + 0.01 * np.random.randn(n_dim) * (np.array(ub) - np.array(lb))
                    x_sample = np.clip(x_sample, lb, ub)

            ##########################################################
            #  X_all, y_all 업데이트 
            X_all = np.vstack([X_all, x_sample])
            y_all = np.append(y_all, y_sample)
            ##########################################################
            
            # 제약조건 체크
            if constraints is not None:
                list_constraints = constraints(x_sample, args)
                if list_constraints and any(c > 0 for c in list_constraints):
                    continue
            
            try:
                y_sample = objective(x_sample, args)
                local_eval_count += 1
                
                if y_sample < best_y:
                    best_y = y_sample
                    best_x = x_sample.copy()
                    no_improve_count = 0
                    print(f"   ⭐ New best at eval {local_eval_count}: {best_y:.4f}")
                else:
                    no_improve_count += 1
                
                if _check_early_stop(best_y, no_improve_count):
                    early_stopped = True
                    break
                
                # Update GP model periodically
                n_data_recent = 100 #150 #50 #100
                if i > 0 and i % 15 == 0 and len(X_all) > 10:
                    try:
                        gp_model.fit(X_all[-n_data_recent:], y_all[-n_data_recent:])  # Use recent data
                    except:
                        #gp_model.fit(X_all, y_all)  # Use recent data
                        pass
                    
            except Exception as e:
                continue
        
        stop_reason = " (early stopped)" if early_stopped else ""
        print(f"✅ Phase 3 complete{stop_reason}. Final best: {best_y:.4f}")
        print(f"🏆 GP-based optimization finished with {local_eval_count} evaluations")
        
        history = {
            'method': 'gp_multi_phase',
            'samples': local_eval_count,
            'kernel': 'valley_specialized' if is_high_dim_intensive else 'common_best',
            'early_stopped': early_stopped if 'early_stopped' in dir() else False,
            'stopped_at': 'phase3' if (early_stopped if 'early_stopped' in dir() else False) else 'completed'
        }
        return best_x, best_y, history
    
    else:
        # 제한된 평가: GP 기반 2단계 최적화
        print(f"🎯 GP-based two-stage optimization (evals: {max_eval_objective} < {threshold})")
        
        # ===================================================================
        # 🔍 STAGE 1: EXPLORATION + GP FITTING (35%)
        # ===================================================================
        stage1_evals = int(max_eval_objective * 0.35)
        stage2_evals = max_eval_objective - stage1_evals
        
        print(f"🔍 Stage 1: Exploration + GP fitting with {stage1_evals} evaluations")
        
        # Stage 1: 넓은 범위에서 다양한 샘플링
        X_stage1 = []
        y_stage1 = []
        current_best_x = None
        current_best_y = np.inf
        early_stopped = False
        
        # Stage 1에서 정확히 stage1_evals 개의 유효한 샘플을 얻을 때까지 반복
        valid_evaluations = 0
        attempt_count = 0
        max_attempts = stage1_evals * 10  # 무한루프 방지
        
        while valid_evaluations < stage1_evals and attempt_count < max_attempts:
            # 다양한 샘플링 전략
            if valid_evaluations < stage1_evals // 3:
                # 완전 랜덤
                x_sample = np.array(lb) + np.random.random(n_dim) * (np.array(ub) - np.array(lb))
            elif valid_evaluations < 2 * stage1_evals // 3:
                # 중심 근처 편향 샘플링
                center = np.array([(lb[j] + ub[j])/2 for j in range(n_dim)])
                x_sample = center + 0.3 * np.random.randn(n_dim) * (np.array(ub) - np.array(lb))
                x_sample = np.clip(x_sample, lb, ub)
            else:
                # 경계 근처 샘플링
                x_sample = np.array(lb) + np.random.random(n_dim) * (np.array(ub) - np.array(lb))
                # 경계쪽으로 편향
                for j in range(n_dim):
                    if np.random.random() < 0.5:
                        x_sample[j] = lb[j] + 0.1 * (ub[j] - lb[j]) * np.random.random()
                    else:
                        x_sample[j] = ub[j] - 0.1 * (ub[j] - lb[j]) * np.random.random()
            
            attempt_count += 1
            
            # 제약조건 체크
            if constraints is not None:
                list_constraints = constraints(x_sample, args)
                if list_constraints and any(c > 0 for c in list_constraints):
                    continue
            
            try:
                y_sample = objective(x_sample, args)
                local_eval_count += 1
                X_stage1.append(x_sample)
                y_stage1.append(y_sample)
                valid_evaluations += 1
                
                if y_sample < current_best_y:
                    current_best_y = y_sample
                    current_best_x = x_sample.copy()
                    no_improve_count = 0
                else:
                    no_improve_count += 1
                
                if _check_early_stop(current_best_y, no_improve_count):
                    early_stopped = True
                    break
                    
            except Exception as e:
                continue
        
        if len(X_stage1) == 0:
            raise ValueError("Stage 1 failed: No valid samples found")
        
        X_stage1 = np.array(X_stage1)
        y_stage1 = np.array(y_stage1)
        best_idx = np.argmin(y_stage1)
        current_best_x = X_stage1[best_idx]
        current_best_y = y_stage1[best_idx]
        
        print(f"Stage 1 complete. Best: {current_best_y:.4f} from {len(X_stage1)} samples")
        
        if early_stopped:
            combined_history = {
                'method': 'gp_two_stage',
                'stage1_evals': len(X_stage1),
                'stage2_evals': 0,
                'total_evals': local_eval_count,
                'kernel': 'valley_specialized' if is_high_dim_intensive else 'common_best',
                'early_stopped': True,
                'stopped_at': 'stage1'
            }
            return current_best_x, current_best_y, combined_history
        
        # ===================================================================
        # 🤖 FIT GP SURROGATE MODEL
        # ===================================================================
        print(f"🤖 Fitting GP surrogate model...")
        try:
            gp_model = GaussianProcessRegressor(
                kernel=kernel,
                alpha=1e-6,
                normalize_y=True, #False,
                n_restarts_optimizer=1,
                random_state=rand_seed
            )
            gp_model.fit(X_stage1, y_stage1)
            print(f"   GP model fitted successfully")
        except Exception as e:
            print(f"   Warning: GP fitting failed, using fallback")
            kernel_fallback = kernel_stable_conservative(n_dim)
            gp_model = GaussianProcessRegressor(
                kernel=kernel_fallback,
                alpha=1e-5,
                normalize_y=False
            )
            gp_model.fit(X_stage1, y_stage1)
        
        # ===================================================================
        # 📊 PROMISING REGION ANALYSIS
        # ===================================================================
        # 상위 30% 점들로 유망 영역 정의
        n_top = max(2, len(y_stage1) * 30 // 100)
        top_indices = np.argsort(y_stage1)[:n_top]
        top_points = X_stage1[top_indices]
        
        # 유망 영역 중심과 범위 계산
        center = np.mean(top_points, axis=0)
        ranges = np.std(top_points, axis=0)
        
        # Stage 2 탐색 범위 설정 (보수적 축소)
        lb_stage2 = []
        ub_stage2 = []
        
        for i in range(n_dim):
            # 최소 범위 보장 (원래 범위의 25%)
            min_range = (ub[i] - lb[i]) * 0.25
            range_i = max(ranges[i] * 2.0, min_range)
            
            lb_new = max(center[i] - range_i, lb[i])
            ub_new = min(center[i] + range_i, ub[i])
            
            lb_stage2.append(lb_new)
            ub_stage2.append(ub_new)
        
        print(f"🎯 Promising region:")
        print(f"   Original: {[f'{x:.2f}' for x in lb]} to {[f'{x:.2f}' for x in ub]}")
        print(f"   Focused:  {[f'{x:.2f}' for x in lb_stage2]} to {[f'{x:.2f}' for x in ub_stage2]}")
        
        # ===================================================================
        # 🎯 STAGE 2: GP-GUIDED INTENSIVE OPTIMIZATION (65%)
        # ===================================================================
        print(f"🎯 Stage 2: GP-guided intensive optimization with {stage2_evals} evaluations")
        
        X_stage2 = []
        y_stage2 = []
        
        # Stage 2에서 정확히 stage2_evals 개의 유효한 샘플을 얻을 때까지 반복
        valid_evaluations = 0
        attempt_count = 0
        max_attempts = stage2_evals * 10  # 무한루프 방지
        
        while valid_evaluations < stage2_evals and attempt_count < max_attempts:
            progress = valid_evaluations / max(1, stage2_evals)
            
            # 40% acquisition-based, 60% local search (속도 최적화)
            if np.random.random() < 0.4:
                try:
                    # Adaptive kappa: 2.0 → 0.3
                    kappa = 2.0 - 1.7 * progress
                    x_sample = optimize_acquisition(
                        gp_model, current_best_y, lb_stage2, ub_stage2,
                        n_restarts=2, acq_type='LCB', kappa=kappa
                    )
                except:
                    # Fallback
                    radius = 0.08 - 0.05 * progress
                    x_sample = current_best_x + radius * np.random.randn(n_dim) * (np.array(ub) - np.array(lb))
                    x_sample = np.clip(x_sample, lb_stage2, ub_stage2)
            else:
                # Local search with decreasing radius
                radius = 0.1 - 0.08 * progress  # 0.1 → 0.02
                direction = np.random.randn(n_dim)
                direction = direction / (np.linalg.norm(direction) + 1e-9)
                x_sample = current_best_x + radius * (np.array(ub) - np.array(lb)) * direction
                x_sample = np.clip(x_sample, lb_stage2, ub_stage2)
            
            attempt_count += 1
            
            # 제약조건 체크
            if constraints is not None:
                list_constraints = constraints(x_sample, args)
                if list_constraints and any(c > 0 for c in list_constraints):
                    continue
            
            try:
                y_sample = objective(x_sample, args)
                local_eval_count += 1
                X_stage2.append(x_sample)
                y_stage2.append(y_sample)
                valid_evaluations += 1
                
                # Update best
                if y_sample < current_best_y:
                    current_best_y = y_sample
                    current_best_x = x_sample.copy()
                    no_improve_count = 0
                    print(f"   ⭐ New best at eval {local_eval_count}: {current_best_y:.4f}")
                else:
                    no_improve_count += 1
                
                if _check_early_stop(current_best_y, no_improve_count):
                    early_stopped = True
                    break
                
                # Update GP periodically (속도 최적화: 덜 자주 업데이트)
                if valid_evaluations > 0 and valid_evaluations % 50 == 0:
                    try:
                        all_X = np.vstack([X_stage1, X_stage2])
                        all_y = np.concatenate([y_stage1, y_stage2])
                        gp_model.fit(all_X, all_y)
                    except:
                        pass
                    
            except Exception as e:
                continue
        
        stop_reason = " (early stopped)" if early_stopped else ""
        print(f"✅ Stage 2 complete{stop_reason}. Best: {current_best_y:.4f}")
        print(f"🏆 GP-based two-stage optimization finished with {local_eval_count} evaluations")
        
        # History
        combined_history = {
            'method': 'gp_two_stage',
            'stage1_evals': len(X_stage1),
            'stage2_evals': len(X_stage2) if X_stage2 else 0,
            'total_evals': local_eval_count,
            'kernel': 'valley_specialized' if is_high_dim_intensive else 'common_best',
            'early_stopped': early_stopped,
            'stopped_at': 'stage2' if early_stopped else 'completed'
        }
        
        return current_best_x, current_best_y, combined_history
        

In [ ]:
rand_seed=12+4+1*np.random.randint(10000)
np.random.seed(rand_seed)

# Suppress sklearn warnings for cleaner output
import warnings
warnings.filterwarnings('ignore', message='Predicted variances smaller than 0.*')
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings('ignore', category=ConvergenceWarning)

# 🎯 Test CAE tools with adaptive optimization
cae_tool_list = ['Goldstein_Price', 'Rosenbrocks_Valley', 'Six_Hump_Camel_Back', 'Cantilever_Beam', 'My_Function']
#cae_tool_list = ['Goldstein_Price', 'Rosenbrocks_Valley', 'Six_Hump_Camel_Back', 'Cantilever_Beam']
#cae_tool_list = ['Cantilever_Beam']
cae_tool_list = ['Cantilever_Beam', 'My_Function']
#cae_tool_list = ['Goldstein_Price', 'Six_Hump_Camel_Back', 'Cantilever_Beam', 'My_Function']
cae_tool_list = ['Goldstein_Price', 'Six_Hump_Camel_Back']

for cae_tool_name in cae_tool_list:
    print("\n=========================================")
    print(f"🔧 Setting up for CAE tool: {cae_tool_name}")
    
    args = [cae_tool_name]

    if cae_tool_name == 'Goldstein_Price':
        n_para = 2
        lb=[-2.0, -2.0]
        ub=[2.0, 2.0]
        x_true = [0, -1]
        y_true = 3
        max_eval_objective = 500 
        n_initials = 35

    elif cae_tool_name == 'Rosenbrocks_Valley':
        n_para = 5
        lb=[-2.048, -2.048, -2.048, -2.048, -2.048]
        ub=[2.048, 2.048, 2.048, 2.048, 2.048]  
        x_true = [1, 1, 1, 1, 1] 
        y_true = 1.0
        max_eval_objective = 1500 
        n_initials = 35
        
    elif cae_tool_name == 'Six_Hump_Camel_Back':
        n_para = 2
        lb=[-1.8, -0.9]
        ub=[1.9, 0.95]   
        x_true = [0.0898, -0.7126]
        y_true = -1.0316
        max_eval_objective = 50  # < 200이므로 2단계 최적화 사용됨
        n_initials = 20

    elif cae_tool_name == 'Cantilever_Beam':
        n_para = 4
        lb=[3.0, 0.1, 2.0, 0.1]
        ub=[7.0, 0.42, 10.2, 0.88]
        x_true = [7.0, 0.1, 9.48482, 0.1] 
        y_true = 92.7707
        max_eval_objective = 150  # < 200이므로 2단계 최적화 사용됨
        n_initials = 5
                
    elif cae_tool_name == 'My_Function':
        n_para = 5 #10
        lb_const=0 # -1 
        ub_const=+1 # +1 

        lb=lb_const*np.ones(n_para)
        ub=ub_const*np.ones(n_para)
                
        x_true = 0.9015*np.ones(n_para)
        y_true = np.sum((-0.8113496993537722*np.ones(n_para)))
        
        max_eval_objective = 1500 #n_para*200 #2*1000  # < 200이므로 2단계 최적화 사용됨
        n_initials = 20 #20
                
    else:
        raise ValueError(f"Unknown CAE model name: {cae_tool_name}")

    # Kernel selection
    kernel = kernel_common_best(n_para)

    # ===================================================================
    # 🎯 GP-BASED ADAPTIVE OPTIMIZATION EXECUTION
    # ===================================================================
    cnt_test = 10 #2 #10  # 테스트용 10회 실행
    list_best_f_ebo = []    
    overall_elapsed_time = 0
    
    for i_test in range(cnt_test):
        start_time = time.time()
        print(f'\ni_test = {i_test+1}/{cnt_test} @ {cae_tool_name}')
        
        # cnt_objective 초기화 (objective 함수에서 사용하는 전역 카운터)
        global cnt_objective
        cnt_objective = 0
        
        # 🚀 GP 기반 스마트 적응형 최적화 실행
        best_x, best_y, history_hybrid = gp_adaptive_optimization(
            objective=objective,
            constraints=constraints,
            kernel=kernel,
            early_stop_y_tolerance=-1e20, #1e-2,  # 목표 y 값 (필요에 따라 조정), -np.inf 로 설정하면 목표 없이 최적화
            early_stop_patience=1e20, #200,   # 연속적으로 개선이 없을 때 조기 종료 (필요에 따라 조정)            
            lb=lb,
            ub=ub,
            args=args,
            max_eval_objective=max_eval_objective,
            threshold=200  # 200 미만인 경우 GP 기반 2단계 최적화 사용
        )
        
        list_best_f_ebo.append(best_y)  
        
        # Cantilever_Beam 제약조건 출력
        if cae_tool_name == 'Cantilever_Beam':
            f_cae, sigma_max, delta_max = cae_Cantilever_Beam(np.array(best_x))
            print(f'   Constraints: sigma_max={sigma_max:.2f} (≤5000), delta_max={delta_max:.4f} (≤0.1)')

        print(f'   Best Result: x_opt={[f"{x:.3f}" for x in best_x]}, y_opt={best_y:.4f}')
        print(f'   True optimal: x_true={x_true}, y_true={y_true}')
        print(f'   Evaluations: {cnt_objective}/{max_eval_objective}')
        print(f'   Elapsed_Time: {time.time() - start_time:.2f}s')
        overall_elapsed_time += time.time() - start_time
    
    # 결과 요약
    mean_best_f_ebo = np.mean(list_best_f_ebo)
    std_best_f_ebo = np.std(list_best_f_ebo)    
    elapsed_time = time.time() - start_time
    
    print(f'\n***** Summary for {cae_tool_name} *****')
    print(f'Max evaluations: {max_eval_objective}')
    adaptive_method = "GP-based Two-stage" if max_eval_objective < 200 else "GP-based Multi-phase"
    print(f'Optimization method: {adaptive_method}')
    print(f'Elapsed time: {elapsed_time:.2f}s')          
    print(f'Mean Best f over {cnt_test} runs: {mean_best_f_ebo:.4f} ± {std_best_f_ebo:.4f}')
    
    # 성공률 계산
    # if cae_tool_name == 'Cantilever_Beam':
    #     target_threshold = y_true * 1.2  # 50% 이내
    #     success_count = sum(1 for f in list_best_f_ebo if f < target_threshold)
    # else:
    #     target_threshold = y_true * 2.0  # 2배 이내
    #     success_count = sum(1 for f in list_best_f_ebo if f < target_threshold)
    if y_true < 0:
        target_threshold = y_true * (1-0.2)  # 20% margin for negative values
        success_count = sum(1 for f in list_best_f_ebo if f < target_threshold)
    elif y_true > 0:
        target_threshold = y_true * 1.2  # 20% margin for positive values
        success_count = sum(1 for f in list_best_f_ebo if f < target_threshold)
    else:  # y_true == 0
        target_threshold = 0.1  # Small threshold for zero case
        success_count = sum(1 for f in list_best_f_ebo if abs(f) < target_threshold)
    success_rate = success_count / cnt_test * 100
    
    print(f'Success rate: {success_rate:.1f}% ({success_count}/{cnt_test} runs within {target_threshold:.1f})')
    
    # 개선도 계산
    if y_true < 0:
        improvement_ratio = mean_best_f_ebo / y_true 
    elif y_true > 0:
        improvement_ratio = y_true / mean_best_f_ebo 
    else:  # y_true == 0
        improvement_ratio = abs(mean_best_f_ebo)
    print(f'Improvement ratio vs. theoretical: {improvement_ratio:.3f} (closer to 1.0 is better)')
    print(f'Overall Elapsed_Time: {overall_elapsed_time:.2f}s')

    print("\n🎉 All optimization tests completed!")


🔧 Setting up for CAE tool: Goldstein_Price

i_test = 1/10 @ Goldstein_Price
📈 GP-based multi-phase optimization (evals: 500 >= 200)
🔍 Phase 1: Initial exploration + GP fitting with 125 evaluations
Phase 1 complete. Best: 27.9574 from 125 samples
🤖 Fitting GP surrogate model...
   GP model fitted successfully
🎯 Phase 2: Acquisition-based exploitation with 225 evaluations
   ⭐ New best at eval 231: 20.0727
   ⭐ New best at eval 249: 4.4294
   ⭐ New best at eval 263: 4.4098
   ⭐ New best at eval 320: 4.2184
Phase 2 complete. Best: 4.2184
🤖 GP model refitted with 350 samples
🔬 Phase 3: GP-guided local refinement with 150 evaluations
   ⭐ New best at eval 427: 3.9165
   ⭐ New best at eval 429: 3.0898
   ⭐ New best at eval 476: 3.0719
   ⭐ New best at eval 490: 3.0563
   ⭐ New best at eval 492: 3.0217
✅ Phase 3 complete. Final best: 3.0217
🏆 GP-based optimization finished with 500 evaluations
   Best Result: x_opt=['0.009', '-1.001'], y_opt=3.0217
   True optimal: x_true=[0, -1], y_true=3
 